# Preprocessing

## 1. Introduction

This notebook prepares the dataset for modelling by applying the necessary cleaning, encoding, and transformation steps identified during the EDA. We address class imbalance, remove leakage-prone features, and convert categorical variables - including "unknown" entries - into model-ready formats. By the end of this stage, the dataset will befully structred, clean, and ready for training.

## 2. Import Libraries and Required Dataset

In [22]:
#Import Required Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from imblearn.combine import SMOTETomek
from imblearn.over_sampling import SMOTE
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from imblearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
#Import Required Dataset
df = pd.read_csv('../01_data/bank-full.csv', sep = ';')
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


## 3. Data Cleaning

### Remove Data Leakage

In [3]:
#Remove Data Leakage
df = df.drop(columns = ['duration'])

### Check for missing values and duplicates

In [4]:
#Confirm no missing values
df.isna().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [5]:
#Confirm no duplicates
print(df.duplicated().sum())

16


In [6]:
#After removing 'duration' column, 16 duplicates are now present
#Removed them as they may distort dataset
df = df.drop_duplicates()

#Check for duplicates again
df.duplicated().sum()

0

## 4. Feature Engineering

### Add Column: Convert `pdays` Column into Binary Feature

In [8]:
#Add a contacted_before column
df['contacted_before'] = (df['pdays'] != -1).astype(int)

### Log Tansformation of Skewed Numeric Variables

In [9]:
#LOG TRANSFORM OF BALANCE
#Shift the dataset by 1 so the log transformation does not break with negative numbers.
min_balance = df['balance'].min()
df['balance_log'] = np.log1p(df['balance'] - min_balance + 1)

In [10]:
#LOG TRANSFORM OF CAMPAIGN
df['campaign_log'] = np.log1p(df['campaign'])

In [11]:
#LOG TRANSFORM OF PREVIOUS
df['previous_log'] = np.log1p(df['previous'])

### Add Column: `age_bucket`

In [12]:
#Add column = age_bucket
df['age_bucket'] = pd.cut(
    df['age'],
    bins = [0,25,55,100],
    labels = ['young', 'middle', 'senior']
)

### Add Column: `balance_positive`

In [13]:
df["balance_positive"] = (df["balance"]>0).astype(int)

### Add Column: `long_campaign`

In [14]:
threshold = 15
df["long_campaign"] = (df["campaign"] > threshold).astype(int)

### Remove the columns balance and campaign
Avoids unnecessary noise.

In [15]:
df = df.drop(columns = ['balance', 'campaign','previous','pdays'])

## 5. Binary and Target Encoding

In [16]:
#List binary columns
binary_cols = ["default","housing","loan"]
df[binary_cols] = (
    df[binary_cols]
    .apply(lambda col: col.map({'yes': 1, 'no': 0, 'unknown': 0}))
)


In [17]:
df["y"] = df["y"].map({'yes':1,'no':0})

In [18]:
#Confirm
df.head()

,age,job,marital,education,default,housing,loan,contact,day,month,poutcome,y,contacted_before,balance_log,campaign_log,previous_log,age_bucket,balance_positive,long_campaign
0,58,management,married,tertiary,0,1,0,unknown,5,may,unknown,0,0,9.226607,0.693147,0.0,senior,1,0
1,44,technician,single,secondary,0,1,0,unknown,5,may,unknown,0,0,8.993427,0.693147,0.0,middle,1,0
2,33,entrepreneur,married,secondary,0,1,1,unknown,5,may,unknown,0,0,8.990068,0.693147,0.0,middle,1,0
3,47,blue-collar,married,unknown,0,1,0,unknown,5,may,unknown,0,0,9.161885,0.693147,0.0,middle,1,0
4,33,unknown,single,unknown,0,0,0,unknown,5,may,unknown,0,0,8.989943,0.693147,0.0,middle,1,0


In [19]:
#Save as a csv
df.to_csv('../01_data/model_df.csv', index = False)

## Full Pipeline

In [23]:
#Define numeric features that need to be scaled and categorical featurs that need OneHotEncoding
numeric_features = ['age', 'balance_log', 'previous_log']
categorical_features = [
    "job","education","marital","contact","poutcome", "day", "month","age_bucket", "default", "housing", "loan"
]

#Create a preprocessing transformer variable
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown = 'ignore'), categorical_features)
    ]
)

#Build the pipeline
pipeline = Pipeline(steps = [
    ('preprocess', preprocessor),
    ('smote', SMOTE(random_state = 42)),
    ('model', LogisticRegression(max_iter = 1000))
])

In [24]:
#SPLIT DATA
X = df.drop("y", axis = 1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

In [25]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['age', 'balance_log',
                                                   'previous_log']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['job', 'education',
                                                   'marital', 'contact',
                                                   'poutcome', 'day', 'month',
                                                   'age_bucket', 'default',
                                                   'housing', 'loan'])])),
                ('smote', SMOTE(random_state=42)),
                ('model', LogisticRegression(max_iter=1000))])

In [26]:
y_pred = pipeline.predict(X_test)

In [27]:
##Evaluation
accuracy_score(y_test,y_pred)

0.7644650956964266

In [28]:
confusion_matrix(y_test, y_pred)

array([[6260, 1721],
       [ 408,  650]], dtype=int64)

In [30]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.94      0.78      0.85      7981
           1       0.27      0.61      0.38      1058

    accuracy                           0.76      9039
   macro avg       0.61      0.70      0.62      9039
weighted avg       0.86      0.76      0.80      9039

